In [1]:
!which python
!python --version

/home/commander-data/Storage/users/mave/.venv/bin/python
Python 3.12.9


In [2]:
import pandas as pd
import pysam
import functions
import gffutils

In [ ]:
gene = {
    "symbol": "NF1",
    "ensembl_gene_id": "ENSG00000196712",
    "chrom": "chr17",
    "start": 31094977,
    "end": 31377675,
    "mane_select": "ENST00000358273", # version 9
    "refseq_select": "NM_001042492", # version 3
    "strand": "+"
}

In [ ]:
# set paths
basepath = "/home/commander-data/Storage/users/mave"
genome_data_path = basepath + "/genome_data"
transcripts_path = genome_data_path + "/ensembl_transcripts/115"
ensembl_gff_db = transcripts_path + '/Homo_sapiens.GRCh38.115.db' # create the database using create_gff_db.py script
genome_path = genome_data_path + "/genomes/grch38.fa"

work_path = basepath + "/search_sequences_300_var"

# define data sources
genome = pysam.FastaFile(genome_path)
ensembl_gff = gffutils.FeatureDB(ensembl_gff_db, keep_order=True)
#[x for x in ensembl_gff.children('transcript:ENST00000358273', featuretype="CDS")]

# extract exons of mane select transcript for gene of interest
exons_oi = None
for exon_id, cds in enumerate(ensembl_gff.children('transcript:' + gene["mane_select"], featuretype="CDS", order_by="start")):
    start = cds.start
    end = cds.end
    chrom = cds.chrom
    new_data = pd.DataFrame([chrom, start, end, exon_id + 1]).T

    if exons_oi is None:
        exons_oi = new_data
    else:
        exons_oi = pd.concat([exons_oi, new_data], axis = 0)
exons_oi.columns = ["chrom", "start", "end", "exon_id"]
exons_oi = exons_oi.set_index("exon_id")

# calculate additional data
exons_oi["length"] = exons_oi["end"] - exons_oi["start"] + 1
exons_oi["cdna_end"] = exons_oi["length"].cumsum()
exons_oi["cdna_start"] = exons_oi["cdna_end"] - exons_oi["length"]
exons_oi["cdna_start"] = exons_oi["cdna_start"] + 1

# extract coding sequence from reference genome
exon_sequences = []
for chrom, start, end in zip(exons_oi["chrom"], exons_oi["start"], exons_oi["end"]):
    exon_sequence = genome.fetch("chr" + chrom, start-1, end)
    exon_sequences.append(exon_sequence)
exons_oi["cdna_seq"] = exon_sequences

cdna = ''.join(exons_oi["cdna_seq"])

exons_oi

,chrom,start,end,length,cdna_end,cdna_start,cdna_seq
exon_id,,,,,,,
1,17,31095310,31095369,60,60,1,ATGGCCGCGCACAGGCCGGTGGAATGGGTCCAGGCCGTGGTCAGCC...
2,17,31155983,31156126,144,204,61,CTTCCAATAAAAACAGGACAGCAGAACACACATACCAAAGTCAGTA...
3,17,31159010,31159093,84,288,205,AGAATATTTGGAGAAGCTGCTGAAAAAAATTTATATCTCTCTCAGT...
4,17,31163186,31163376,191,479,289,CAACCAAAGGACACAATGAGATTAGATGAAACGATGCTGGTCAAAC...
5,17,31169891,31169997,107,586,480,GTTACAGGAATTAACTGTTTGTTCAGAAGACAATGTTGATGTTCAT...
6,17,31181422,31181489,68,654,587,AAACAGCATTTAAATTTAAAGCCCTAAAGAAGGTTGCGCAGTTAGC...
7,17,31181710,31181785,76,730,655,GCATTTTGGAACTGGGTAGAAAATTATCCAGATGAATTTACAAAAC...
8,17,31182508,31182665,158,888,731,AATGTGCAGAAAAGCTATTTGACTTGGTGGATGGTTTTGCTGAAAG...
9,17,31200422,31200595,174,1062,889,AAGTTATTTCTGGACAGTCTACGAAAAGCTCTTGCTGGCCATGGAG...


In [ ]:
library_file = work_path + "/data/input/THE_FINAL_LIBRARY.tsv"
library = pd.read_csv(library_file, sep = "\t")
library


,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,Spacer_sequence_order_TOP,Spacer_sequence_order_BOTTOM,pegRNA_extension_sequence_order_TOP,pegRNA_extension_sequence_order_BOTTOM,RHA_length,longest_T_stretch_spacer,longest_T_stretch_extension,longest_T_stretch,complete_peg_rna,gene
0,chr17-31225098-G-A,chr17-31225106-A-G,31225098.0,0.10,43.0,-0.18,152.0,3.0,3.0,152.0,...,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGaCAGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCTGtC,22.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1
1,chr17-31225098-G-A,chr17-31225109-T-C,31225098.0,0.07,43.0,-0.04,-3.0,3.0,3.0,152.0,...,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGaCAGATAGAAGcTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAgCTTCTATCTGtC,19.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1
2,chr17-31225098-G-T,chr17-31225106-A-G,31225098.0,0.18,43.0,-0.29,152.0,3.0,3.0,152.0,...,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGtCAGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCTGaC,22.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1
3,chr17-31225098-G-T,chr17-31225109-T-C,31225098.0,0.19,43.0,-0.17,0.0,3.0,3.0,152.0,...,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcGtCAGATAGAAGcTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAgCTTCTATCTGaC,19.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1
4,chr17-31225100-A-T,chr17-31225106-A-G,31225100.0,0.19,41.0,-0.07,-5.0,5.0,5.0,150.0,...,caccGTAAAAAAGGAGAAAGTGACgtttt,ctctaaaacGTCACTTTCTCCTTTTTTAC,gtgcCtGATAGgAGTTCCTGTCACTTTCTCCTTT,aaaaAAAGGAGAAAGTGACAGGAACTcCTATCaG,22.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,chr17-31233133-G-T,old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TGATCGGTTTGAGAGATTGGGTTTTAGAGCTAGAAATAGCAAGTTA...,NF1
337,chr17-31233072-A-G,old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CTGACAAAAATCCTTCAACAGTTTTAGAGCTAGAAATAGCAAGTTA...,NF1
338,chr17-31233123-G-A,old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CAGAAACAGTATTGGCTGATGTTTTAGAGCTAGAAATAGCAAGTTA...,NF1
339,chr17-31233156-T-G,old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GTCACAATGATGGGTGATCAGTTTTAGAGCTAGAAATAGCAAGTTA...,NF1


In [ ]:
result = []
for mut_oi, syn_mut in zip(library["var_id"], library["syn_var"]):
    if syn_mut == "old":
        continue
    mut_oi = mut_oi.split("-")
    syn_mut = syn_mut.split('-')
    combined_variant = functions.get_mutated_sequence([mut_oi,syn_mut], genome, format = "alt", flanking_dist = 10)
    genomic_wt = combined_variant[2]
    genomic_mut = combined_variant[3]

    cdna_mut = mut_oi.copy()
    cdna_mut[1] = functions.genomic2cdna(exons_oi, int(cdna_mut[1]))
    cdna_syn = syn_mut.copy()
    cdna_syn[1] = functions.genomic2cdna(exons_oi, int(cdna_syn[1]))
    #print(str(cdna_mut) + " " + str(cdna_syn))

    cdna_start = min([cdna_mut[1], cdna_syn[1]]) - 10
    cdna_end = max([cdna_mut[1], cdna_syn[1]]) + 10
    cdna_wt = cdna[cdna_start - 1:cdna_end]

    cdna_mut_seq = cdna_wt
    offset = 0
    last_pos = None
    for mutation in [cdna_mut, cdna_syn]:
        pos = int(mutation[1])
        ref = mutation[2]
        alt = mutation[3]
        if last_pos is None or last_pos < pos:
            cdna_mut_seq, offset = functions.introduce_mutation(cdna_mut_seq, (pos + offset) - cdna_start + 1, ref, alt, format = "alt")
        else:
            cdna_mut_seq, offset = functions.introduce_mutation(cdna_mut_seq, pos - cdna_start + 1, ref, alt, format = "alt")
        last_pos = pos
    
    result.append({
        "var_id": '-'.join(mut_oi),
        "syn_var": '-'.join(syn_mut),
        "genomic_wt": genomic_wt,
        "genomic_mut": genomic_mut,
        "cdna_wt": cdna_wt,
        "cdna_mut": cdna_mut_seq,
    })

result = pd.DataFrame(result)
result

,var_id,syn_var,genomic_wt,genomic_mut,cdna_wt,cdna_mut
0,chr17-31225098-G-A,chr17-31225106-A-G,TTTCTAGCAGGCAGATAGAAGTTCCTGTC,TTTCTAGCAGACAGATAGGAGTTCCTGTC,AAATAAGCAGGCAGATAGAAGTTCCTGTC,AAATAAGCAGACAGATAGGAGTTCCTGTC
1,chr17-31225098-G-A,chr17-31225109-T-C,TTTCTAGCAGGCAGATAGAAGTTCCTGTCACT,TTTCTAGCAGACAGATAGAAGCTCCTGTCACT,AAATAAGCAGGCAGATAGAAGTTCCTGTCACT,AAATAAGCAGACAGATAGAAGCTCCTGTCACT
2,chr17-31225098-G-T,chr17-31225106-A-G,TTTCTAGCAGGCAGATAGAAGTTCCTGTC,TTTCTAGCAGTCAGATAGGAGTTCCTGTC,AAATAAGCAGGCAGATAGAAGTTCCTGTC,AAATAAGCAGTCAGATAGGAGTTCCTGTC
3,chr17-31225098-G-T,chr17-31225109-T-C,TTTCTAGCAGGCAGATAGAAGTTCCTGTCACT,TTTCTAGCAGTCAGATAGAAGCTCCTGTCACT,AAATAAGCAGGCAGATAGAAGTTCCTGTCACT,AAATAAGCAGTCAGATAGAAGCTCCTGTCACT
4,chr17-31225100-A-T,chr17-31225106-A-G,TCTAGCAGGCAGATAGAAGTTCCTGTC,TCTAGCAGGCTGATAGGAGTTCCTGTC,ATAAGCAGGCAGATAGAAGTTCCTGTC,ATAAGCAGGCTGATAGGAGTTCCTGTC
...,...,...,...,...,...,...
327,chr17-31336865-C-T,chr17-31336856-T-C,CTTCCACACATGGACTGGTCATTAATATCA,CTTCCACACACGGACTGGTTATTAATATCA,CTTCCACACATGGACTGGTCATTAATATCA,CTTCCACACACGGACTGGTTATTAATATCA
328,chr17-31336865-C-T,chr17-31336868-T-A,ATGGACTGGTCATTAATATCATTC,ATGGACTGGTTATAAATATCATTC,ATGGACTGGTCATTAATATCATTC,ATGGACTGGTTATAAATATCATTC
329,chr17-31336867-T-G,chr17-31336860-C-T,CACACATGGACTGGTCATTAATATCATT,CACACATGGATTGGTCAGTAATATCATT,CACACATGGACTGGTCATTAATATCATT,CACACATGGATTGGTCAGTAATATCATT
330,chr17-31336867-T-G,chr17-31336865-C-G,ATGGACTGGTCATTAATATCATT,ATGGACTGGTGAGTAATATCATT,ATGGACTGGTCATTAATATCATT,ATGGACTGGTGAGTAATATCATT


In [ ]:
# pull the search sequences from the previous screen
ex27_constanze = pd.read_csv(work_path + "/data/input/ex27_constanze_pegrnas.tsv", sep = "\t")
ex27_constanze = ex27_constanze[["id", "Search sequence gDNA", "WT sequence gDNA", "Search sequence cDNA", "WT sequence cDNA"]]
ex27_constanze.columns = ["id", "genomic_mut", "genomic_wt", "cdna_mut", "cdna_wt"]

constanze_id2myID = [
    {"var_id": "chr17-31233016-A-T", "id": "STOP_29560034"},
    {"var_id": "chr17-31233128-T-A", "id": "STOP_29560146"},
    {"var_id": "chr17-31233005-T-G", "id": "PDA_078_L1167*"},
    {"var_id": "chr17-31233157-C-T", "id": "SD7357_Q1218*"},
    {"var_id": "chr17-31233133-G-T", "id": "P-0011260-T01-IM5_E1210*"},
    {"var_id": "chr17-31233072-A-G", "id": "Synonymous_edit_29560090"},
    {"var_id": "chr17-31233123-G-A", "id": "Synonymous_edit_29560141"},
    {"var_id": "chr17-31233156-T-G", "id": "TCGA-RD-A8N1-01_D1217E"},
    {"var_id": "chr17-31233126-A-G", "id": "Synonymous_edit_29560144"},
]
constanze_id2myID = pd.DataFrame(constanze_id2myID)

ex27_constanze = ex27_constanze.merge(constanze_id2myID, on = "id")
del ex27_constanze["id"]
ex27_constanze["syn_var"] = "old"
ex27_constanze

,genomic_mut,genomic_wt,cdna_mut,cdna_wt,var_id,syn_var
0,ACATTAGGCTGAGGCTACCACAAGG,ACATTAGGCTTAGGTTACCACAAGG,CTCCATAGGCTGAGGCTACCACAAGG,CTCCATAGGCTTAGGTTACCACAAGG,chr17-31233005-T-G,old
1,AGGTTACCACTAGGACCTCCAGACAA,AGGTTACCACAAGGATCTCCAGACAA,AGGTTACCACTAGGACCTCCAGACAA,AGGTTACCACAAGGATCTCCAGACAA,chr17-31233016-A-T,old
2,TCCTTCAACAGGGAACAGAATTTG,TCCTTCAACAAGGCACAGAATTTG,TCCTTCAACAGGGAACAGAATTTG,TCCTTCAACAAGGCACAGAATTTG,chr17-31233072-A-G,old
3,ATCGGTTTGAAAGGTTGGTGGAAC,ATCGGTTTGAGAGATTGGTGGAAC,ATCGGTTTGAAAGGTTGGTGGAAC,ATCGGTTTGAGAGATTGGTGGAAC,chr17-31233123-G-A,old
4,GGTTTGAGAGGTTAGTGGAACTGG,GGTTTGAGAGATTGGTGGAACTGG,GGTTTGAGAGGTTAGTGGAACTGG,GGTTTGAGAGATTGGTGGAACTGG,chr17-31233126-A-G,old
5,TTTGAGAGATAGGTTGAACTGGTCA,TTTGAGAGATTGGTGGAACTGGTCA,TTTGAGAGATAGGTTGAACTGGTCA,TTTGAGAGATTGGTGGAACTGGTCA,chr17-31233128-T-A,old
6,GAGATTGGTGTAACTTGTCACAATGA,GAGATTGGTGGAACTGGTCACAATGA,GAGATTGGTGTAACTTGTCACAATGA,GAGATTGGTGGAACTGGTCACAATGA,chr17-31233133-G-T,old
7,TGATGGGTGAGCAGGGAGAACTCC,TGATGGGTGATCAAGGAGAACTCC,TGATGGGTGAGCAGGGAGAACTCC,TGATGGGTGATCAAGGAGAACTCC,chr17-31233156-T-G,old
8,GATGGGTGATTAAGGGGAACTCCCTA,GATGGGTGATCAAGGAGAACTCCCTA,GATGGGTGATTAAGGGGAACTCCCTA,GATGGGTGATCAAGGAGAACTCCCTA,chr17-31233157-C-T,old


In [ ]:
# collect the results
result = pd.concat([result,ex27_constanze])
library = library.merge(result, on = ["var_id", "syn_var"])
library["exon_number"] = library["exon_number"].fillna(27)
library

,var_id,syn_var,start_combined,pangolin_gain_score_combined,pangolin_gain_pos_combined,pangolin_loss_score_combined,pangolin_loss_pos_combined,distance_to_ss_combined,distance_to_ss_left_combined,distance_to_ss_right_combined,...,RHA_length,longest_T_stretch_spacer,longest_T_stretch_extension,longest_T_stretch,complete_peg_rna,gene,genomic_wt,genomic_mut,cdna_wt,cdna_mut
0,chr17-31225098-G-A,chr17-31225106-A-G,31225098.0,0.10,43.0,-0.18,152.0,3.0,3.0,152.0,...,22.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1,TTTCTAGCAGGCAGATAGAAGTTCCTGTC,TTTCTAGCAGACAGATAGGAGTTCCTGTC,AAATAAGCAGGCAGATAGAAGTTCCTGTC,AAATAAGCAGACAGATAGGAGTTCCTGTC
1,chr17-31225098-G-A,chr17-31225109-T-C,31225098.0,0.07,43.0,-0.04,-3.0,3.0,3.0,152.0,...,19.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1,TTTCTAGCAGGCAGATAGAAGTTCCTGTCACT,TTTCTAGCAGACAGATAGAAGCTCCTGTCACT,AAATAAGCAGGCAGATAGAAGTTCCTGTCACT,AAATAAGCAGACAGATAGAAGCTCCTGTCACT
2,chr17-31225098-G-T,chr17-31225106-A-G,31225098.0,0.18,43.0,-0.29,152.0,3.0,3.0,152.0,...,22.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1,TTTCTAGCAGGCAGATAGAAGTTCCTGTC,TTTCTAGCAGTCAGATAGGAGTTCCTGTC,AAATAAGCAGGCAGATAGAAGTTCCTGTC,AAATAAGCAGTCAGATAGGAGTTCCTGTC
3,chr17-31225098-G-T,chr17-31225109-T-C,31225098.0,0.19,43.0,-0.17,0.0,3.0,3.0,152.0,...,19.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1,TTTCTAGCAGGCAGATAGAAGTTCCTGTCACT,TTTCTAGCAGTCAGATAGAAGCTCCTGTCACT,AAATAAGCAGGCAGATAGAAGTTCCTGTCACT,AAATAAGCAGTCAGATAGAAGCTCCTGTCACT
4,chr17-31225100-A-T,chr17-31225106-A-G,31225100.0,0.19,41.0,-0.07,-5.0,5.0,5.0,150.0,...,22.0,1.0,3.0,3.0,GTAAAAAAGGAGAAAGTGACgttttagagctagaaatagcaagtta...,NF1,TCTAGCAGGCAGATAGAAGTTCCTGTC,TCTAGCAGGCTGATAGGAGTTCCTGTC,ATAAGCAGGCAGATAGAAGTTCCTGTC,ATAAGCAGGCTGATAGGAGTTCCTGTC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336,chr17-31233133-G-T,old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,TGATCGGTTTGAGAGATTGGGTTTTAGAGCTAGAAATAGCAAGTTA...,NF1,GAGATTGGTGGAACTGGTCACAATGA,GAGATTGGTGTAACTTGTCACAATGA,GAGATTGGTGGAACTGGTCACAATGA,GAGATTGGTGTAACTTGTCACAATGA
337,chr17-31233072-A-G,old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,CTGACAAAAATCCTTCAACAGTTTTAGAGCTAGAAATAGCAAGTTA...,NF1,TCCTTCAACAAGGCACAGAATTTG,TCCTTCAACAGGGAACAGAATTTG,TCCTTCAACAAGGCACAGAATTTG,TCCTTCAACAGGGAACAGAATTTG
338,chr17-31233123-G-A,old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,CAGAAACAGTATTGGCTGATGTTTTAGAGCTAGAAATAGCAAGTTA...,NF1,ATCGGTTTGAGAGATTGGTGGAAC,ATCGGTTTGAAAGGTTGGTGGAAC,ATCGGTTTGAGAGATTGGTGGAAC,ATCGGTTTGAAAGGTTGGTGGAAC
339,chr17-31233156-T-G,old,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,GTCACAATGATGGGTGATCAGTTTTAGAGCTAGAAATAGCAAGTTA...,NF1,TGATGGGTGATCAAGGAGAACTCC,TGATGGGTGAGCAGGGAGAACTCC,TGATGGGTGATCAAGGAGAACTCC,TGATGGGTGAGCAGGGAGAACTCC


In [ ]:
# write search sequences to file(s)
for exon_oi in library["exon_number"].unique():
    exon_oi = int(exon_oi)
    for dna_or_rna in ["DNA", "RNA"]:
        if dna_or_rna == "DNA":
            ss_col = "genomic_mut"
        elif dna_or_rna == "RNA":
            ss_col = "cdna_mut"
        search_sequences = library[(library["exon_number"] == exon_oi)][["combined_var_id", ss_col]]
        #search_sequences[ss_col + "_rv"] = [x.translate(str.maketrans("ACGTacgt", "TGCAtgca"))[::-1] for x in search_sequences[ss_col]]

        ss_file_path = work_path + "/data/input/search_sequences/ss_E" + str(exon_oi) + "_" + dna_or_rna + ".txt"
        search_sequences.to_csv(ss_file_path, sep = " ", index = False, header = False)
